# Accessing OpenNeuro Datasets in Neurodesk

**Author**: Michèle Masson-Trottier

The University of Queensland<br>
<div style="line-height: 2;">
<a href="https://github.com/micmas"><img src="https://img.shields.io/badge/-Michèle_Masson--Trottier-181717?logo=github" alt="GitHub"></a> <a href="https://orcid.org/0000-0002-0642-5662"><img src="https://img.shields.io/badge/ORCID-0000--0002--0642--5662-green?logo=orcid" alt="ORCID"></a>
</div>

**Date**: 30/03/2026

**License:** 
<div style="margin-top: 10px;">
    <a href="https://creativecommons.org/licenses/by/4.0/" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> CC-BY-4.0
    </a>
</div>

## Purpose

OpenNeuro is the largest open repository of neuroimaging data in BIDS format. This tutorial covers three methods for accessing OpenNeuro datasets from within Neurodesk: using the DataLad interface (recommended), using the AWS S3 interface, and using the openneuro-py Python client.

:::{admonition} Learning Objectives
:class: tip
By the end of this tutorial you will be able to:
- Browse and identify datasets on OpenNeuro
- Use DataLad to lazily clone and fetch OpenNeuro datasets
- Download specific files using the AWS S3 CLI
- Use openneuro-py for programmatic dataset access
- Understand BIDS folder structure for OpenNeuro datasets
:::

## Citation and Resources

### Tools and datasets used in this workflow

__OpenNeuro__
: Markiewicz, C.J., et al. (2021). The OpenNeuro resource for sharing of neuroscience data. *eLife*, 10, e71774. [https://doi.org/10.7554/eLife.71774](https://doi.org/10.7554/eLife.71774)

__DataLad__
: Halchenko, Y., et al. (2021). DataLad: distributed system for joint management of code, data, and their relationship. *Journal of Open Source Software*, 6(63), 3262.

### Educational resources

- [OpenNeuro website](https://openneuro.org)
- [DataLad documentation](https://www.datalad.org)
- [Neurodesk documentation](https://neurodesk.org)

## Prerequisites

:::{admonition} Before you begin
:class: warning
Make sure you have access to a running Neurodesk instance. See [Getting Set Up with Neurodesk](https://neurodesk.org/getting-started/) for instructions.
:::

- [x] A running Neurodesk environment
- [ ] Familiarity with basic terminal commands
- [ ] Internet connection for accessing OpenNeuro
- [ ] Access to `~/neurodesktop-storage/` directory

## Overview: What is OpenNeuro?

OpenNeuro is the largest open repository of neuroimaging data in BIDS format, hosting thousands of datasets across multiple modalities including fMRI, diffusion MRI, EEG, MEG, and structural imaging. All datasets on OpenNeuro are curated to be BIDS-compliant and published with open access (CC0 or CC-BY-4.0 license).

Each dataset receives a persistent DOI for citation, making it easy to reference in publications. Datasets are identified by unique identifiers like `ds000102`.

![The OpenNeuro website showing the dataset browser.](/static/tutorials/open_data/openneuro/openneuro_website.png)
*The OpenNeuro website showing the dataset browser at https://openneuro.org*

## Browsing Datasets

Before downloading a dataset, you can browse available datasets directly on the OpenNeuro website. Navigate to [https://openneuro.org](https://openneuro.org) to search and filter datasets.

**Search features:**
- **Search bar** — search by dataset name, description, or keywords
- **Filters** — filter by modality (fMRI, dMRI, EEG, etc.), task, age range, species, and more
- **Dataset page** — each dataset shows metadata, participant information, acquisition parameters, and download options

Dataset identifiers follow the format `dsXXXXXXX` (e.g., `ds000102`). All datasets have persistent DOIs for citation.

![An OpenNeuro dataset page showing metadata and download options.](/static/tutorials/open_data/openneuro/openneuro_dataset_page.png)
*An OpenNeuro dataset page showing metadata, participant demographics, and available download methods.*

## Method 1: DataLad (Recommended)

DataLad is the recommended way to access OpenNeuro datasets because it allows **lazy fetching** — you clone the dataset metadata instantly (just a few MB), then only download the files you actually need. This saves bandwidth and time.

### Installing DataLad

DataLad is pre-installed in most Neurodesk environments. If not, install it via:

```bash
pip install datalad
```

### Cloning a Dataset

To clone a dataset (e.g., `ds000102`), navigate to `~/neurodesktop-storage/` and run:

```bash
cd ~/neurodesktop-storage/

# Install (clone) the dataset — downloads only metadata (~MB, not the data itself)
datalad install https://github.com/OpenNeuroDatasets/ds000102.git
```

This creates a `ds000102/` directory with full dataset metadata. The actual data files are not yet downloaded.

![Terminal output after datalad install showing dataset metadata.](/static/tutorials/open_data/openneuro/terminal_datalad_install.png)
*Terminal output after datalad install showing the cloned dataset structure.*

### Fetching Specific Files

Once the dataset is cloned, use `datalad get` to download only the files you need:

```bash
cd ds000102

# Download a single file
datalad get sub-08/anat/sub-08_T1w.nii.gz

# Download all data for a single subject
datalad get sub-08/

# Download all data for all subjects (use with caution — can be very large!)
# datalad get .
```

![Terminal showing datalad get downloading a specific file.](/static/tutorials/open_data/openneuro/terminal_datalad_get.png)
*Terminal showing datalad get downloading a specific anatomical file.*

## Method 2: AWS S3

OpenNeuro datasets are hosted on Amazon S3. If you prefer not to use DataLad, you can download files directly using the AWS S3 CLI. No AWS account is required for public datasets — they use the `--no-sign-request` flag.

### Installing the AWS CLI

Load the AWS CLI tools in Neurodesk:

```bash
ml awscli
```

### Downloading a Single File

```bash
# Download a single file
aws s3 cp \
    s3://openneuro.org/ds000102/sub-08/anat/sub-08_T1w.nii.gz \
    ~/neurodesktop-storage/ds000102_s3/T1w.nii.gz \
    --no-sign-request
```

### Syncing an Entire Folder

```bash
# Sync all data for a subject
aws s3 sync \
    s3://openneuro.org/ds000102/sub-08/ \
    ~/neurodesktop-storage/ds000102_s3/sub-08/ \
    --no-sign-request
```

This method is useful for downloading a subset of files without needing DataLad, but DataLad is generally more efficient for repeated access to the same dataset.

## Method 3: openneuro-py

The `openneuro-py` Python client provides a programmatic interface for downloading datasets. This is useful for automating data retrieval in Python workflows.

### Installing openneuro-py

```bash
pip install openneuro-py
```

### Downloading Specific Files

```python
import openneuro

# Download specific files from a dataset
openneuro.download(
    dataset="ds000102",
    target_dir="~/neurodesktop-storage/ds000102_py/",
    include=["sub-08/anat/sub-08_T1w.nii.gz"]
)
```

### Listing Available Files

```python
import openneuro

# Get metadata about a dataset
ds_info = openneuro.get_dataset("ds000102")
print(ds_info)
```

This method is ideal for building reproducible scripts that automatically fetch data as part of a larger analysis pipeline.

## Understanding the BIDS Structure

After downloading an OpenNeuro dataset, the files are organised according to the **Brain Imaging Data Structure (BIDS)** standard. Understanding this structure helps you locate the files you need.

### Typical BIDS Layout

```
ds000102/
├── sub-01/
│   ├── anat/
│   │   ├── sub-01_T1w.nii.gz
│   │   └── sub-01_T1w.json
│   ├── func/
│   │   ├── sub-01_task-rest_bold.nii.gz
│   │   └── sub-01_task-rest_bold.json
│   └── dwi/
│       ├── sub-01_dwi.nii.gz
│       ├── sub-01_dwi.bval
│       └── sub-01_dwi.bvec
├── sub-02/
│   └── ...
├── dataset_description.json
├── participants.tsv
└── README
```

**Key components:**
- **`sub-XX/`** — subject directories
- **`anat/`** — anatomical images (T1w, T2w, etc.)
- **`func/`** — functional images (fMRI)
- **`dwi/`** — diffusion-weighted images
- **`dataset_description.json`** — metadata about the dataset
- **`participants.tsv`** — demographic information (age, sex, group, etc.)

![File browser showing BIDS folder structure of a downloaded OpenNeuro dataset.](/static/tutorials/open_data/openneuro/bids_folder_structure.png)
*File browser showing the BIDS folder structure of a downloaded OpenNeuro dataset.*

### Reading Participant Information

The `participants.tsv` file contains demographic data in tab-separated format. You can read it using pandas:

```python
import pandas as pd

# Load participant information
participants = pd.read_csv(
    '~/neurodesktop-storage/ds000102/participants.tsv',
    sep='\t'
)

# Display first few rows
print(participants.head())

# Summary statistics
print(participants.describe())
```

This allows you to filter subjects by age, group, or other criteria for your analysis.

## Summary

In this tutorial you:

1. Learned about OpenNeuro and its role as a BIDS-compliant neuroimaging data repository
2. Browsed and identified datasets on openneuro.org
3. Downloaded datasets using three methods: DataLad (recommended), AWS S3, and openneuro-py
4. Understood the BIDS folder structure and how to extract metadata from downloaded datasets

**Key takeaway:** DataLad is the recommended approach for most users because it enables efficient, lazy downloading of only the files you need.

:::{seealso}
- [DataLad tutorial](datalad.ipynb) — for advanced DataLad workflows and reproducible data management
- [DataLad-run tutorial](datalad-run.ipynb) — for automating analyses with DataLad
- [OSF client tutorial](osfclient.ipynb) — for accessing data on the Open Science Framework
:::